# 02. Prompt Engineering 실습
**SK하이닉스 Autonomous R&D — Day 3** 

---
## 0. 준비

In [1]:
from openai import OpenAI
from dotenv import load_dotenv
import os

load_dotenv("../../.env")

client = OpenAI(api_key=os.getenv('OPENAI_KEY'))


def ask(system_prompt, user_prompt, temperature=0.2):
    response = client.chat.completions.create(
        model='gpt-4o-mini',
        temperature=temperature,
        messages=[
            {'role': 'system', 'content': system_prompt},
            {'role': 'user', 'content': user_prompt},
        ],
    )
    return response.choices[0].message.content

In [2]:
# 기본적으로 같은 모델이라도 프롬프트에 따라 다른 출력을 생성

system = 'You are a helpful assistant. Answer in Korean.'

prompts_ko = [
    "프랑스의 수도는",
    "Q: 프랑스의 수도는 어디인가요? A:",
    "프랑스의 수도 도시는",
]

for prompt in prompts_ko:
    print(f"프롬프트: {prompt}")
    print(f"생성: {ask(system, prompt, temperature=0.2)}")
    print()

프롬프트: 프랑스의 수도는
생성: 프랑스의 수도는 파리입니다.

프롬프트: Q: 프랑스의 수도는 어디인가요? A:
생성: 프랑스의 수도는 파리입니다.

프롬프트: 프랑스의 수도 도시는
생성: 프랑스의 수도 도시는 파리입니다.



---
## 1. Zero-shot vs Few-shot 

| 방식 | 설명 |
|------|------|
| **Zero-shot** | 예시 없이 지시만 |
| **Few-shot** | 원하는 형식의 **예시**를 함께 제공 |

In [3]:
system = 'You are a helpful assistant. Answer in Korean.'
# Zero-shot — 형식 지시 없음
user_zero = '''
아래 영화 리뷰가 긍정인지 부정인지 판정해줘.
- "연출은 좋았지만 2시간이 너무 길었다"
- "최고의 영화, 또 보고 싶다"
'''
print('=== Zero-shot ===')
print(ask(system, user_zero))

=== Zero-shot ===
첫 번째 리뷰는 부정적인 요소가 포함되어 있어 부정으로 판정할 수 있습니다. 두 번째 리뷰는 긍정적인 감정을 표현하고 있어 긍정으로 판정할 수 있습니다.


In [4]:
# Few-shot — 출력 형식 예시 포함
user_few = '''
아래 형식으로 감성 판정해줘.

[예시]
입력: "배우 연기가 훌륭했다" → 긍정 | 9/10
입력: "스토리가 지루했다" → 부정 | 3/10

[실제 데이터]
- "연출은 좋았지만 2시간이 너무 길었다"
- "최고의 영화, 또 보고 싶다"
'''
print('=== Few-shot ===')
print(ask(system, user_few))

=== Few-shot ===
입력: "연출은 좋았지만 2시간이 너무 길었다" → 중립 | 5/10  
입력: "최고의 영화, 또 보고 싶다" → 긍정 | 10/10


In [5]:
zero_shot_prompt_ko = """다음은 단어 유추의 예시입니다:

문제: 행복한 : 슬픈 :: 좋은 :"""

print(ask(system, zero_shot_prompt_ko, temperature=0.7))

좋은 : 나쁜


In [6]:
few_shot_prompt_ko = """다음은 단어 유추의 예시입니다:

뜨거운 : 차가운 :: 위 : 아래
큰 : 작은 :: 빠른 : 느린
낮 : 밤 :: 밝은 : 어두운
행복한 : 슬픈 :: 좋은 :"""

print('Few-shot 프롬프트:')
print(few_shot_prompt_ko)
print('\n생성된 완성:')
print(ask(system, few_shot_prompt_ko, temperature=0.7))

Few-shot 프롬프트:
다음은 단어 유추의 예시입니다:

뜨거운 : 차가운 :: 위 : 아래
큰 : 작은 :: 빠른 : 느린
낮 : 밤 :: 밝은 : 어두운
행복한 : 슬픈 :: 좋은 :

생성된 완성:
나쁜


---
## 2. 역할(Role) 설정 

In [7]:
question = '하루 커피 4잔 마셔도 괜찮을까?'

print('=== 일반 assistant ===')
print(ask('You are a helpful assistant.', question))
print('\n' + '=' * 50 + '\n')

=== 일반 assistant ===
하루에 커피 4잔을 마시는 것은 대부분의 사람들에게 괜찮습니다. 일반적으로 카페인 섭취에 대한 권장량은 하루 400mg 이하로, 이는 대략 커피 4잔에 해당합니다. 하지만 개인의 카페인 민감도, 건강 상태, 임신 여부 등에 따라 다를 수 있습니다.

카페인에 민감한 사람은 적은 양으로도 불안, 불면증, 심박수 증가 등의 증상을 경험할 수 있으므로, 자신의 몸 상태를 잘 살펴보는 것이 중요합니다. 또한, 커피 외에도 다른 음료나 음식에서 카페인을 섭취할 수 있으니 전체적인 카페인 섭취량을 고려하는 것이 좋습니다.

혹시 특정한 건강 문제가 있거나 카페인 섭취에 대해 걱정이 된다면, 의사와 상담하는 것이 좋습니다.




In [ ]:
print('=== 영양사 역할 ===')
print(ask(
    'You are a registered dietitian. Answer in Korean with caffeine mg estimate and health advice.',
    question,
))

In [8]:
roles = [
    "당신은 친절한 고객 서비스 상담원입니다.",
    "당신은 전문적인 기술 문서 작성자입니다.",
    "당신은 창의적인 소설가입니다."
]

question = "인공지능에 대해 설명해주세요."

for role in roles:
    user_prompt = f"{role}\n\n질문: {question}\n\n답변:"
    print(f"=== {role} ===")
    print(f"답변: {ask('Answer in Korean.', user_prompt, temperature=0.8)}")
    print()

=== 당신은 친절한 고객 서비스 상담원입니다. ===
답변: 안녕하세요! 인공지능(AI)은 기계나 컴퓨터가 인간처럼 사고하고 학습할 수 있도록 하는 기술입니다. 인공지능은 다양한 형태로 존재하며, 주로 데이터 분석, 패턴 인식, 자연어 처리, 이미지 인식 등 여러 분야에서 활용됩니다.

AI는 크게 두 가지 유형으로 나눌 수 있습니다. 첫째, 좁은 인공지능(Narrow AI)은 특정 작업을 수행하도록 설계된 시스템으로, 예를 들어 음성 인식 소프트웨어나 추천 시스템이 이에 해당합니다. 둘째, 일반 인공지능(General AI)은 인간과 유사한 수준의 지능을 가지고 다양한 작업을 수행할 수 있는 시스템을 의미하지만, 현재로서는 연구 단계에 머물러 있습니다.

인공지능의 발전은 우리의 삶을 더욱 편리하게 만들고 있으며, 의료, 자동차, 금융 등 다양한 산업에서 혁신을 이끌고 있습니다. 추가적인 질문이 있으시면 언제든지 말씀해 주세요!

=== 당신은 전문적인 기술 문서 작성자입니다. ===
답변: 인공지능(AI, Artificial Intelligence)은 컴퓨터 시스템이 인간과 유사한 인지 기능을 수행할 수 있도록 설계된 기술을 의미합니다. 이는 학습(머신러닝), 추론, 문제 해결, 인식, 자연어 처리 등 다양한 기능을 포함합니다. 인공지능은 주로 두 가지 주요 카테고리로 나눌 수 있습니다: 

1. **약한 인공지능(Weak AI)**: 특정 작업에 특화된 시스템으로, 일반적인 지능을 갖추고 있지 않습니다. 예를 들어, 음성 인식 소프트웨어나 자율 주행차의 경우가 여기에 해당합니다.

2. **강한 인공지능(Strong AI)**: 인간의 지능 수준에 해당하는 지능을 목표로 하는 시스템으로, 다양한 작업을 수행하고 스스로 학습하며 이해할 수 있는 능력을 가지고 있습니다. 현재로서는 이론적인 개념에 가깝고, 실제 구현은 이루어지지 않았습니다.

인공지능의 발전은 머신러닝, 심층 학습(딥러닝), 자연어 처리(NLP) 등의 기술을 통해 이루어지고 있으며, 의료,

---
## 3. 출력 형식 지정 

In [9]:
topic = '재택근무의 장단점 3가지'
system_ko = 'Answer in Korean.'

print('=== Markdown ===')
print(ask(system_ko, topic + '\n형식: markdown bullet point'))
print('\n' + '=' * 50 + '\n')

=== Markdown ===
### 재택근무의 장단점

#### 장점
- **유연한 근무 시간**: 개인의 일정에 맞춰 근무 시간을 조정할 수 있어 일과 삶의 균형을 맞추기 용이함.
- **교통비 및 시간 절약**: 출퇴근이 필요 없어 교통비와 시간을 절약할 수 있음.
- **편안한 근무 환경**: 자신이 선호하는 환경에서 일할 수 있어 집중력과 생산성이 향상될 수 있음.

#### 단점
- **사회적 고립감**: 동료와의 직접적인 소통이 줄어들어 외로움을 느낄 수 있음.
- **업무와 개인 생활의 경계 모호**: 집에서 일하다 보면 업무와 개인 생활의 경계가 흐려질 수 있음.
- **자기 관리의 어려움**: 자율성이 높아지면서 스스로 동기를 부여하고 관리하는 것이 어려울 수 있음.




In [10]:
print('=== Table ===')
print(ask(
    system_ko,
    topic + '\n형식: markdown 표 (항목 | 장점 | 단점)만 출력',
))

=== Table ===
| 항목       | 장점                           | 단점                           |
|------------|--------------------------------|--------------------------------|
| 유연성     | 근무 시간을 자유롭게 조정 가능 | 일과 개인 생활의 경계 모호함  |
| 교통비 절감| 출퇴근 시간과 비용 절약       | 사회적 상호작용 부족          |
| 생산성     | 집중할 수 있는 환경 조성 가능  | 자칫 산만해질 수 있음        |


---
### 출력 형식 (표 / JSON)

In [ ]:
import json

topic = '식각(Etch) 공정에서 수율에 영향 주는 요인 3가지'
system_ko = 'Answer in Korean.'

print('=== Table ===')
print(ask(
    system_ko,
    topic + '\n형식: markdown 표 (요인 | 설명 | 대응)만 출력',
))

In [ ]:
print('=== JSON ===')
result = ask(
    'Output ONLY valid JSON. No markdown.',
    topic + '\n{"factors": [{"name": "", "action": ""}]} 형식으로',
)
print(result)
try:
    print('\n파싱 성공:', list(json.loads(result).keys()))
except json.JSONDecodeError:
    print('\n파싱 실패 — ONLY valid JSON 강조 필요')

---
## 4. Chain-of-Thought (CoT)

CoT는 모델이 단계별로 추론 과정을 보여주도록 하는 기법.

In [11]:
## zero-shot 예시

problem = '''
A팀 10명, B팀 15명, C팀 5명.
전체 25명 중 40% 이상이 A팀이면 A팀에 추가 인원 배치.
지금 추가 배치가 필요한가?
'''
system = 'You are a helpful assistant. Answer in Korean.'
print('=== CoT 없음 ===')
print(ask(system, problem))

=== CoT 없음 ===
전체 인원 25명 중 40%는 10명입니다. 현재 A팀은 10명으로, 전체 인원의 40%에 해당합니다. 따라서 A팀에 추가 배치가 필요하지 않습니다. A팀의 인원이 40%를 초과하지 않기 때문에 추가 배치가 필요하지 않습니다.


In [12]:
print('=== CoT 적용 ===')
print(ask(
    system + ' Show step-by-step reasoning before final answer.',
    problem + '\n\n단계별로 생각해 봅시다.',
))

=== CoT 적용 ===
전체 인원은 25명입니다. A팀, B팀, C팀의 인원 수는 다음과 같습니다:

- A팀: 10명
- B팀: 15명
- C팀: 5명

1. **A팀 인원의 비율 계산**:
   A팀의 인원 수는 10명입니다. 전체 인원 25명 중 A팀의 비율을 계산해보겠습니다.
   \[
   \text{A팀 비율} = \frac{\text{A팀 인원}}{\text{전체 인원}} = \frac{10}{25} = 0.4
   \]
   즉, A팀의 비율은 40%입니다.

2. **40% 이상인지 확인**:
   문제에서 요구하는 조건은 A팀의 인원이 전체 인원의 40% 이상이어야 한다고 했습니다. 현재 A팀의 비율은 정확히 40%입니다.

3. **추가 배치 필요성 판단**:
   A팀의 인원이 40% 이상이므로, 추가 인원 배치가 필요하지 않습니다. 

결론적으로, A팀에 추가 배치가 필요하지 않습니다.


In [ ]:
problem = """다음 처럼 문제를 답하시오.

문제: 한 상점에서 연필 5자루와 지우개 3개를 샀습니다. 연필은 개당 500원, 지우개는 개당 300원입니다. 총 금액은 얼마인가요?

답: ?원"""

system = 'You are a helpful assistant. Answer in Korean.'

print(ask(system, problem))

In [ ]:
problem = """다음 처럼 문제를 답하시오.

문제: 한 상점에서 연필 5자루와 지우개 3개를 샀습니다. 연필은 개당 500원, 지우개는 개당 300원입니다. 총 금액은 얼마인가요?

천천히 단계별로 생각해봅시다.

답: ?원"""

system = 'You are a helpful assistant. Answer in Korean.'

print(ask(system, problem))

In [ ]:
problem = """다음 처럼 문제를 답하시오.

문제: 한 상점에서 사과 3개와 배 2개를 샀습니다. 사과는 개당 1000원, 배는 개당 1500원입니다. 총 금액은 얼마인가요?

답: 6000원

---

문제: 한 상점에서 귤 1개와 바나나 12개를 샀습니다. 사과는 개당 1000원, 배는 개당 200원입니다. 총 금액은 얼마인가요?

답: 3400원

---


문제: 한 상점에서 연필 5자루와 지우개 3개를 샀습니다. 연필은 개당 500원, 지우개는 개당 300원입니다. 총 금액은 얼마인가요?

답: """

system = 'You are a helpful assistant. Answer in Korean.'

print(ask(system, problem))

In [ ]:
## one-shot 예시

problem = """다음 문제를 단계별로 풀어보세요.

문제: 한 상점에서 사과 3개와 배 2개를 샀습니다. 사과는 개당 1000원, 배는 개당 1500원입니다. 총 금액은 얼마인가요?

단계별 풀이:
1. 사과 3개의 가격: 3 × 1000 = 3000원
2. 배 2개의 가격: 2 × 1500 = 3000원
3. 총 금액: 3000 + 3000 = 6000원
답: 6000원

---

문제: 한 상점에서 연필 5자루와 지우개 3개를 샀습니다. 연필은 개당 500원, 지우개는 개당 300원입니다. 총 금액은 얼마인가요?

단계별 풀이:"""

system = 'You are a helpful assistant. Answer in Korean.'

print(ask(system, problem))

---
## 5. System + User 프롬프트 

In [ ]:
system_prompt = '''
너는 스타트업 PM의 일정 관리 AI다.
규칙: 결론 먼저, bullet 3개, 한국어
'''

user_prompt = '''
이번 주 마감: 월-기획안, 수-코드리뷰, 금-발표.
목요일에 하루 종일 회의가 잡혔어. 우선순위 조정해줘.
'''

print(ask(system_prompt, user_prompt))

In [ ]:
system_md = (
    "너는 백화점 MD 업무를 보조하는 AI다. "
    "답변은 반드시 표 형태로 작성하고, "
    "결론을 먼저 제시한 뒤 "
    "리스크와 가정을 명시해야 한다."
)

user_md = "이번 주 VIP 이탈 상위 200명에 대한 액션 플랜을 작성해줘."

answer = ask(system_md, user_md, temperature=0.7)
print(answer)

---
## 6. Self-check Prompting

In [ ]:
check_prompt = f"""아래는 네가 작성한 'VIP 이탈 상위 200명 액션 플랜' 초안이다.

[초안]
{answer}

이제 아래 체크리스트로 점검하고, 필요하면 수정본을 작성하라.

체크리스트:
1) 논리적 모순/비약이 있는가?
2) 실행 불가능하거나 모호한 액션이 있는가? (담당/기한/방법이 불명확한지)
3) 과도한 가정이 있는가?
4) 표 형식이 지켜졌는가? (결론 먼저, 리스크/가정 포함)

출력 규칙:
- 먼저 "점검 결과"를 bullet로 간단히 쓰고
- 그 다음 "수정본"을 작성하라
- 수정본은 반드시 표 형태 + 결론 먼저 + 리스크/가정 명시
"""

final_answer = ask(system_md, check_prompt, temperature=0.7)
print('\n--- Self-check 후 최종 답변(수정본) ---')
print(final_answer)

---
## 7. Constraint Prompting

In [ ]:
system_prompt = (
    "너는 백화점 MD 업무를 보조하는 AI다. "
    "답변은 반드시 표 형태로 작성하고, "
    "결론을 먼저 제시한 뒤, "
    "리스크와 가정을 명시해야 한다."
)

constraints = """조건(반드시 준수):
- 추가 예산은 최대 1억 원 이내
- 인력 증원 불가 (기존 인력 내에서 운영)
- 2주 이내 실행 가능한 액션만 제시
- 고객에게 직접 연락하는 액션은 과도한 빈도를 피하고(1회 이내),
  개인정보/컴플라이언스 리스크를 고려할 것
"""

user_prompt = f"""이번 주 VIP 이탈 상위 200명에 대한 액션 플랜을 작성해줘.

{constraints}

출력 형식:
- 결론(한 줄) 먼저
- 그 다음 표로 정리 (컬럼 예: 타깃/액션/채널/우선순위/기간/기대효과/담당)
- 마지막에 리스크/가정 명시
"""

constrained_answer = ask(system_prompt, user_prompt, temperature=0.7)
print('\n--- 조건 기반 최종 답변(답변만) ---')
print(constrained_answer)